[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sear-labs/energy-system-modeling/blob/main/notebooks/p1_foundations/01_model_boundary.ipynb)
# Where do you draw the boundary?
## REE 4301 / IE 5300 - Energy Systems Modeling
### Do this straight after the toy LP notebook. Before anything else.

In Module 0 you built a model of a whole system and it told you a price: the shadow price on the balance constraint. **You produced that number.**

Almost none of you will be paid to do that.

You will be paid to work on **one facility** - a warehouse, a campus, a plant, a data centre - and for that job the price is not something you compute. It is a column in a CSV you download. Same number. Opposite direction.

> **In a macro model, price is an output.**
> **In a facility model, price is an input.**

That sentence is the whole of this notebook, and it is the most useful thing in the course for the first job you take. Everything after this is working out **when you are allowed to believe it.**


## Setup

One cell, the same in every notebook here: it installs what Colab does not have, fetches this repository so `data/` and `src/` are present, and moves into this notebook's own folder so the relative paths below resolve. The notebook's own imports follow in the same cell.


In [ ]:
# --- setup: generated by tools/sync_setup_cells.py -- do not edit here, edit that
# The same cell in every notebook in this series. It installs what Colab does
# not have, fetches the repository so that data/ and src/ are present, and moves
# into this notebook's own folder so the relative paths below resolve.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sear-labs/energy-system-modeling"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1]
NOTEBOOK_DIR = "notebooks/p1_foundations"

# Pinned, per Part 1 rule 3: an unpinned install will one day pull a major
# version with a changed API and either break or silently alter the answer.
PINS = ["gurobipy>=11,<14", "highspy>=1.11,<2", "pypsa>=1.3,<2"]

if "google.colab" in sys.modules:
    if PINS:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    # An ABSOLUTE base, so re-running this cell is safe. Colab's "Run all" is
    # commonly run twice, and a relative check would look for the clone inside
    # the folder it had already moved into -- cloning a second copy nested one
    # level down, then working from the wrong one.
    BASE = Path("/content") if Path("/content").is_dir() else Path.home()
    REPO_DIR = BASE / REPO_NAME
    if not REPO_DIR.exists():
        cloned = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            capture_output=True, text=True)
        if cloned.returncode != 0:
            raise SystemExit(
                "Could not clone " + REPO_URL + "\n"
                + (cloned.stderr or "").strip() + "\n\n"
                "If that says 'not found', the repository is still private.\n"
                "A raw file URL fails the same way, so there is no way around\n"
                "it: it has to be public before a student can run this.")
    os.chdir(REPO_DIR / NOTEBOOK_DIR)
elif Path.cwd().name != Path(NOTEBOOK_DIR).name:
    raise SystemExit(
        "Run this notebook from its own folder (" + NOTEBOOK_DIR + "),\n"
        "so that ../../src and ../../data resolve.")

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
print("working directory:", Path.cwd().name)

# --- end generated setup; the notebook's own imports follow ---



import numpy as np
import pandas as pd
import gurobipy as gp
import pypsa
import warnings
warnings.filterwarnings('ignore')

SOLVER = 'gurobi'
# SOLVER = 'highs'     # <-- uncomment: open source, no licence, no size cap

WLS = {}   # {'WLSACCESSID': '...', 'WLSSECRET': '...', 'LICENSEID': 000000}
ENV = gp.Env(params=WLS) if (SOLVER == 'gurobi' and WLS) else None

print(f'solver: {SOLVER}'
      + ('  (academic WLS licence)' if ENV else '  (default licence)'))

---
# Part A - One site, two lives

**The Metroplex Industrial Park** sits north of Dallas: warehousing, some light manufacturing, a cold-storage tenant. It draws **50 MW**. It has an interconnection agreement with the utility for **600 MW** - generous, because the site was master-planned for heavy industry that never fully arrived.

In 2027 an investment group buys the site. The land is cheap, the fibre is already there, and - the reason they actually bought it - **the interconnection agreement already exists.** Getting a new one takes years. They convert the park into a hyperscale data centre drawing **500 MW**.

Same land. Same substation. Same agreement. Ten times the load.

**Nothing about the site's location changed. The correct model changed.** Working out why, and what it costs you to get it wrong, is the rest of this notebook.


### Two questions that look the same and are not

| | the macro question | the facility question |
|---|---|---|
| who asks it | ERCOT, a regulator, a developer siting a fleet | you, on one site |
| what is decided | what gets built and dispatched everywhere | what this site builds and when it runs |
| **price is** | **an output** - a dual variable | **an input** - a downloaded series |
| demand is | forecast for a region | metered, and partly yours to shift |
| the answer | a system plan | a capital decision and a bill |

The vocabulary for the second row, which you should use in an interview:

- A facility that is too small to move the price is a **price taker**. One that is big enough to move it is a **price maker**.
- Moving price from something the model solves for to something you hand it is moving it from **endogenous** to **exogenous**.
- Running a big model once to produce inputs for a small model is **soft-linking**. Solving both together is **hard-linking**.
- The facility problem itself has a name: **price-taker dispatch optimisation**.


---
# Part B - The macro model, which produces a price

First build the thing you will *not* normally be asked to build, so you know what the number you download actually is.

A Dallas node with a metro load of about 4,000 MW, a lot of wind, and a merit order of gas plants from cheap to ruinous. This is a deliberately small stand-in for ERCOT, but the shape is right: cheap units for most hours, and a thin, very expensive tail for the few hours a year when everything is needed at once.


### Step 1 - the hours, the load, and the wind


In [2]:
hours = np.arange(24)

# metro demand: overnight base, morning bump, evening peak
metro = (4000
         + 1000 * np.exp(-((hours - 19) ** 2) / 12.0)
         + 400 * np.exp(-((hours - 8) ** 2) / 6.0))

# west Texas wind: strong overnight and through the morning, gone by dusk
wind_pu = np.clip(0.30 + 0.55 * np.sin(np.pi * (hours - 2) / 16), 0, 1)

print(pd.DataFrame({'metro MW': metro.round(0),
                    'wind p.u.': wind_pu.round(2)}).T.to_string())


                0        1       2        3        4        5        6        7        8        9        10       11       12       13       14       15       16       17      18       19       20      21      22      23
metro MW   4000.00  4000.00  4001.0  4006.00  4028.00  4089.00  4205.00  4339.00  4400.00  4339.00  4207.00  4094.00  4045.00  4056.00  4126.00  4264.00  4472.00  4717.00  4920.0  5000.00  4920.00  4717.0  4472.0  4264.0
wind p.u.     0.09     0.19     0.3     0.41     0.51     0.61     0.69     0.76     0.81     0.84     0.85     0.84     0.81     0.76     0.69     0.61     0.51     0.41     0.3     0.19     0.09     0.0     0.0     0.0


### Step 2 - the merit order

Eight 600 MW units from $22 to $90, then a thin tail: two 300 MW peakers at $175 and $600, and 300 MW of last-resort capacity at $2,000.

**That tail is not invented.** ERCOT's offer cap has been as high as $5,000/MWh. Almost every hour of the year the price is set near the bottom of a stack like this; a handful of hours are set near the top, and those hours are where the money is - and where a large new load does its damage.


In [3]:
STACK = [(f'gas{i + 1:02d}', 600, c) for i, c in
         enumerate([22., 26., 31., 37., 44., 52., 62., 90.])]
STACK += [('peaker1', 300, 175.), ('peaker2', 300, 600.),
          ('scarcity', 300, 2000.)]

for name, mw, cost in STACK:
    print(f'  {name:9s} {mw:5.0f} MW  ${cost:8,.0f}/MWh')
print(f'  {"firm total":9s} {sum(m for _, m, _ in STACK):5.0f} MW')


  gas01       600 MW  $      22/MWh
  gas02       600 MW  $      26/MWh
  gas03       600 MW  $      31/MWh
  gas04       600 MW  $      37/MWh
  gas05       600 MW  $      44/MWh
  gas06       600 MW  $      52/MWh
  gas07       600 MW  $      62/MWh
  gas08       600 MW  $      90/MWh
  peaker1     300 MW  $     175/MWh
  peaker2     300 MW  $     600/MWh
  scarcity    300 MW  $   2,000/MWh
  firm total  5700 MW


### Step 3 - the site's own bus, which is the boundary

Here is the modelling idea worth keeping. **The site gets its own Bus, and a Link to the grid.**

That Link is the interconnection agreement, and it is the meter. Everything on the grid side of it is somebody else's problem; everything on the site side is yours. **The boundary is not a concept in this model - it is a component you can point at.**

`p_nom=600` is the agreement itself. Remember that number.

**One note on style.** This is wrapped in a function, which the rest of this course avoids — you build models a component at a time so you can see each decision. It is wrapped here for one reason: we are about to solve this same network a dozen times at different site sizes, and you have already built every one of these components by hand in Module 0. The rule has not changed — wrap it *after* you understand it, never before.


In [4]:
def grid_model(site_mw):
    """The macro model. Returns the network, or None if it cannot solve."""
    n = pypsa.Network()
    n.set_snapshots(hours)

    n.add('Bus', 'Dallas')       # the grid
    n.add('Bus', 'Site')         # everything behind the meter

    n.add('Generator', 'wind', bus='Dallas', p_nom=5000,
          marginal_cost=0, p_max_pu=pd.Series(wind_pu, index=hours))
    for name, mw, cost in STACK:
        n.add('Generator', name, bus='Dallas', p_nom=mw, marginal_cost=cost)

    n.add('Load', 'metro', bus='Dallas',
          p_set=pd.Series(metro, index=hours))

    # the interconnection agreement: 600 MW, and it is the model boundary
    n.add('Link', 'interconnection', bus0='Dallas', bus1='Site',
          p_nom=600, efficiency=1.0)
    if site_mw > 0:
        n.add('Load', 'site', bus='Site', p_set=float(site_mw))

    status, condition = n.optimize(solver_name=SOLVER, env=ENV,
                                   log_to_console=False)
    return n if condition == 'optimal' else None


### Step 4 - run it with no site at all, and read the price

This is the world before the investors show up. The `marginal_price` column is the same dual variable you met in Module 0 - **this is the number that gets published, and the number a facility analyst downloads.**


In [5]:
n_base = grid_model(0)
lmp_base = n_base.buses_t.marginal_price['Dallas']

print(pd.DataFrame({'metro MW': metro.round(0),
                    'wind p.u.': wind_pu.round(2),
                    'LMP $/MWh': lmp_base.round(2)}).to_string())
print()
print(f'cheapest hour ${lmp_base.min():,.2f}')
print(f'dearest hour  ${lmp_base.max():,.2f}')


Index(['Dallas', 'Site'], dtype='object', name='name')


Index(['interconnection'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.04s


...

INFO:gurobipy:Solved in 22 iterations and 0.01 seconds (0.00 work units)


INFO:gurobipy:Optimal objective  1.669377755e+06


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 312 primals, 672 duals
Objective: 1.67e+06
Solver: gurobi
Runtime: 0.01s
Dual bound: 1.67e+06
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper were not assigned to the network.


          metro MW  wind p.u.  LMP $/MWh
snapshot                                
0           4000.0       0.09       52.0
1           4000.0       0.19       52.0
2           4001.0       0.30       44.0
3           4006.0       0.41       37.0
4           4028.0       0.51       31.0
5           4089.0       0.61       26.0
6           4205.0       0.69       26.0
7           4339.0       0.76       22.0
8           4400.0       0.81       22.0
9           4339.0       0.84       22.0
10          4207.0       0.85        0.0
11          4094.0       0.84        0.0
12          4045.0       0.81       22.0
13          4056.0       0.76       22.0
14          4126.0       0.69       26.0
15          4264.0       0.61       31.0
16          4472.0       0.51       37.0
17          4717.0       0.41       44.0
18          4920.0       0.30       52.0
19          5000.0       0.19       62.0
20          4920.0       0.09       90.0
21          4717.0       0.00       90.0
22          4472

---
# Part C - The industrial park, and the assumption that works

You are the analyst for the 50 MW park. You do **not** build the model above - you would not have the data, and nobody is paying you to model ERCOT. You download `lmp_base` and multiply.

**Predict first:** the park is 50 MW arriving on a node already serving 4,000 MW. Do you think its own arrival changes the price it pays?


In [6]:
# the facility view: price is an INPUT. This is the whole calculation.
park_predicted = (lmp_base * 50.0).sum()

print(f'park load                 50 MW, flat')
print(f'predicted day-ahead bill  ${park_predicted:,.0f} /day')
print(f'                          ${park_predicted * 365 / 1e6:,.1f} M/yr')


park load                 50 MW, flat
predicted day-ahead bill  $49,500 /day
                          $18.1 M/yr


### Now check it against the macro model

Put the park into the grid model and re-solve. If the exogenous-price assumption is sound, the price should barely move.


In [7]:
n_park = grid_model(50)
lmp_park = n_park.buses_t.marginal_price['Dallas']

park_actual = (lmp_park * 50.0).sum()
moved = int((~np.isclose(lmp_park, lmp_base)).sum())
err = (park_actual / park_predicted - 1) * 100

print(f'hours in which the park moved the price : {moved} of 24')
print(f'predicted bill  ${park_predicted:>10,.0f}')
print(f'actual bill     ${park_actual:>10,.0f}')
print(f'error           {err:>10.1f} %')


Index(['Dallas', 'Site'], dtype='object', name='name')


Index(['interconnection'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.03s


...

INFO:gurobipy:Solved in 23 iterations and 0.01 seconds (0.00 work units)


INFO:gurobipy:Optimal objective  1.719053702e+06


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 312 primals, 672 duals
Objective: 1.72e+06
Solver: gurobi
Runtime: 0.01s
Dual bound: 1.72e+06
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper were not assigned to the network.


hours in which the park moved the price : 3 of 24
predicted bill  $    49,500
actual bill     $    51,300
error                  3.6 %


**Three hours out of twenty-four, and a 3.6% error on the bill.**

For a screening study, a lease negotiation, or a board paper, that is fine. You would spend more than 3.6% of the answer arguing about the weather year. **The park is a price taker, and treating the price as exogenous is the correct professional judgement.**

Notice what you did *not* have to do: model ERCOT, forecast a generation fleet, or know anything about West Texas wind. That is what lowering the boundary buys you.


---
# Part D - The data centre, and the same assumption failing

The investors close. The site becomes a **500 MW** data centre on the same interconnection.

A new analyst - or the same one, on autopilot - does the same calculation. Downloads the published prices, multiplies by 500.

**Predict before running.** The load went up by a factor of ten. Does the error go up by a factor of ten, or by more, or by less?


In [8]:
dc_predicted = (lmp_base * 500.0).sum()

n_dc = grid_model(500)
lmp_dc = n_dc.buses_t.marginal_price['Dallas']
dc_actual = (lmp_dc * 500.0).sum()

moved_dc = int((~np.isclose(lmp_dc, lmp_base)).sum())
err_dc = (dc_actual / dc_predicted - 1) * 100

print(f'hours in which the data centre moved the price : {moved_dc} of 24')
print(f'predicted bill  ${dc_predicted:>10,.0f} /day')
print(f'actual bill     ${dc_actual:>10,.0f} /day')
print(f'error           {err_dc:>10.1f} %')
print()
print(f'annual gap      ${(dc_actual - dc_predicted) * 365 / 1e6:,.0f} M/yr')


Index(['Dallas', 'Site'], dtype='object', name='name')


Index(['interconnection'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.03s


...

INFO:gurobipy:Solved in 24 iterations and 0.01 seconds (0.00 work units)


INFO:gurobipy:Optimal objective  2.323468350e+06


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 312 primals, 672 duals
Objective: 2.32e+06
Solver: gurobi
Runtime: 0.01s
Dual bound: 2.32e+06
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper were not assigned to the network.


hours in which the data centre moved the price : 19 of 24
predicted bill  $   495,000 /day
actual bill     $   912,000 /day
error                 84.2 %

annual gap      $152 M/yr


In [9]:
compare = pd.DataFrame({
    'metro MW': metro.round(0),
    'LMP alone': lmp_base.round(2),
    'LMP + park': lmp_park.round(2),
    'LMP + data centre': lmp_dc.round(2),
})
print(compare.to_string())


          metro MW  LMP alone  LMP + park  LMP + data centre
snapshot                                                    
0           4000.0       52.0        62.0               62.0
1           4000.0       52.0        52.0               52.0
2           4001.0       44.0        44.0               52.0
3           4006.0       37.0        37.0               44.0
4           4028.0       31.0        31.0               37.0
5           4089.0       26.0        26.0               31.0
6           4205.0       26.0        26.0               31.0
7           4339.0       22.0        26.0               26.0
8           4400.0       22.0        22.0               26.0
9           4339.0       22.0        22.0               26.0
10          4207.0        0.0        22.0               22.0
11          4094.0        0.0         0.0               22.0
12          4045.0       22.0        22.0               22.0
13          4056.0       22.0        22.0               26.0
14          4126.0      

### Read hour 21

The price goes from **$90 to $600**. The data centre pushed the system off the shoulder of the merit order and into the scarcity tail - and then paid $600/MWh for **all 500 MW**, in an hour that only cost $90 before it existed.

This is the whole failure mode. The price-taker calculation was not slightly optimistic. **It answered a different question**: what the site would pay in a world where the site does not exist.

### And it is not only the data centre's problem


In [10]:
metro_extra_dc = ((lmp_dc - lmp_base) * metro).sum()
metro_extra_park = ((lmp_park - lmp_base) * metro).sum()

print(f'extra cost to the existing 4,000 MW metro load:')
print(f'  because of the park        ${metro_extra_park:>12,.0f} /day')
print(f'  because of the data centre ${metro_extra_dc:>12,.0f} /day')
print()
print(f'the data centre pays ${dc_actual:,.0f}/day for itself,')
print(f'and imposes ${metro_extra_dc:,.0f}/day on everyone else -')
print(f'a factor of {metro_extra_dc / dc_actual:.1f} times its own bill.')


extra cost to the existing 4,000 MW metro load:
  because of the park        $     149,898 /day
  because of the data centre $   3,878,864 /day

the data centre pays $912,000/day for itself,
and imposes $3,878,864/day on everyone else -
a factor of 4.3 times its own bill.


> **Exercise D.1.** The data centre imposes more cost on its neighbours than it pays for its own electricity. Nothing it did was against the rules. Who should pay for that, and what would each answer do to where data centres get built?
>
> This is not a hypothetical. It is the live argument in ERCOT, PJM and half a dozen other markets right now, and if you interview at a utility, a developer or a regulator in the next two years, some version of it will come up.
>
> **Exercise D.2.** The park's error was 3.6% and the data centre's is 84%. The load only grew by 10x. Explain the non-linearity using the merit-order table from Part B.


---
# Part E - So where exactly is the line?

"Price taker" is not a property of a facility. It is a property of a **facility and a system together**, and it degrades continuously. Find out where it stops being safe.


In [11]:
rows = []
for mw in [10, 25, 50, 100, 200, 300, 500, 600]:
    nk = grid_model(mw)
    if nk is None:
        rows.append({'site MW': mw, 'predicted $/day': None,
                     'actual $/day': None, 'error %': None,
                     'verdict': 'WILL NOT SOLVE'})
        continue
    lk = nk.buses_t.marginal_price['Dallas']
    pred = (lmp_base * mw).sum()
    act = (lk * mw).sum()
    e = (act / pred - 1) * 100
    rows.append({'site MW': mw,
                 'predicted $/day': round(pred),
                 'actual $/day': round(act),
                 'error %': round(e, 1),
                 'verdict': 'price taker' if e < 5 else
                            ('borderline' if e < 20 else 'PRICE MAKER')})

print(pd.DataFrame(rows).set_index('site MW').to_string())


Index(['Dallas', 'Site'], dtype='object', name='name')


Index(['interconnection'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.03s


...

INFO:gurobipy:Solved in 24 iterations and 0.01 seconds (0.00 work units)


INFO:gurobipy:Optimal objective  2.512276297e+06


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 312 primals, 672 duals
Objective: 2.51e+06
Solver: gurobi
Runtime: 0.01s
Dual bound: 2.51e+06
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper were not assigned to the network.


         predicted $/day  actual $/day  error %      verdict
site MW                                                     
10                  9900          9900      0.0  price taker
25                 24750         24750      0.0  price taker
50                 49500         51300      3.6  price taker
100                99000        111100     12.2   borderline
200               198000        235200     18.8   borderline
300               297000        354000     19.2   borderline
500               495000        912000     84.2  PRICE MAKER
600               594000       1160400     95.4  PRICE MAKER


> **There is no bright line, and that is the honest answer.** Somewhere between 50 and 200 MW on this system, a facility stops being a spectator and starts being a participant. Where exactly depends on the shape of the supply stack, not on the facility.
>
> **The rule to carry:** treating price as exogenous is a *modelling assumption with an error bar*, not a fact. Test it by putting your load into a system model once. If the price barely moves, you never have to do it again. If it moves, you have just learned that your project is big enough to need a seat at a different table.


### And the harder limit

The interconnection agreement is 600 MW. The investors, encouraged, propose a second phase taking the site to **900 MW**.


In [12]:
n_big = grid_model(900)

if n_big is None:
    print('the model will not solve at 900 MW.')
    print()
    print('This is not a bug and not a numerical problem. The site is')
    print('asking for 900 MW through a 600 MW agreement. No dispatch')
    print('exists that satisfies every constraint, so the LP correctly')
    print('reports that there is no answer.')
    print()
    print('An infeasible model has told you something true: this phase')
    print('does not happen without a new interconnection study, new')
    print('equipment, and years of queue time. That is the single')
    print('largest risk in a project like this, and the model found it')
    print('before anyone signed anything.')
else:
    print('solved - check the interconnection p_nom')


Index(['Dallas', 'Site'], dtype='object', name='name')


Index(['interconnection'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.03s


...

INFO:gurobipy:


INFO:gurobipy:Solved in 0 iterations and 0.00 seconds (0.00 work units)


INFO:gurobipy:Infeasible or unbounded model


Status: warning
Termination condition: infeasible_or_unbounded
Solution: 0 primals, 0 duals
Objective: nan
Solver: gurobi
Runtime: 0.01s
Solver model: available
Solver message: 4



the model will not solve at 900 MW.

This is not a bug and not a numerical problem. The site is
asking for 900 MW through a 600 MW agreement. No dispatch
exists that satisfies every constraint, so the LP correctly
reports that there is no answer.

An infeasible model has told you something true: this phase
does not happen without a new interconnection study, new
equipment, and years of queue time. That is the single
largest risk in a project like this, and the model found it
before anyone signed anything.


---
# Part F - What actually crosses the boundary

Lowering the boundary is **not** "ignore the grid". It is "import the grid as a boundary condition". For a facility, exactly four things cross that line:

**1. A price signal.** A nodal LMP series or a retail tariff. This is the one you have been using all notebook.

**2. Coincident-peak exposure.** In ERCOT this is **4CP**: a large customer's transmission charge for the whole next year is set by its demand during the four 15-minute intervals that turn out to be the monthly system peaks in June, July, August and September. Four intervals. If you can predict them and shed load, you avoid a substantial annual charge. **Note what that requires: forecasting the grid's peak, not your own.** This is the clearest case of a facility needing to understand the system it sits in - for four hours a year.

**3. Interconnection capacity and queue position.** Part E, and usually the binding constraint on whether a project happens at all.

**4. Marginal emissions intensity.** If anyone has promised 24/7 carbon-free energy, the number that matters is the *marginal* emissions of the hour you consume in, not the annual average of the grid.

Everything else stays outside. You do not need to know the LMP a hundred miles away, what gets built in 2035, or how a refinery plans its turnaround - unless one of those four channels carries it to you.


### The same five modules, at the lower boundary

Every model in this course survives the boundary change. Only the question changes:

| module | at the macro scale | at your facility |
|---|---|---|
| **1 Demand** | forecast a region's load | your load is *metered*, and partly yours to shift |
| **2 Generation** | what capacity should the system build | should *this site* build solar, storage, or sign a PPA |
| **3 Networks** | where does congestion bind, what should be reinforced | your interconnection limit and your tariff |
| **4 Storage** | system arbitrage and reliability | demand-charge and 4CP management |
| **5 Supply chain** | where should refining capacity go | delivered cost and single-source risk at your dock |

Each later module in this course ends with a short **"at the facility scale"** note that makes this concrete. When you get there, the machinery will already be familiar - the only thing that moves is which quantities are given to you and which you get to choose.


---
## What you should take from this

1. **Choosing the boundary is the first modelling decision**, and it is yours to make and to defend. Everything downstream inherits it.
2. **Price is an output of a macro model and an input to a facility model.** Same dual variable, opposite direction.
3. **Price taking is an assumption with an error bar**, not a property of your project. Here it cost 3.6% at 50 MW and 84% at 500 MW.
4. **Test it once.** Put your load in a system model and see whether the price moves. That single run is the difference between a defensible number and a confident wrong one.
5. **Infeasible is an answer.** The 900 MW phase failed against the interconnection agreement, which is exactly what an interconnection study exists to find out.

### If someone asks you in an interview what you can do

> *"I can take a site's load and an hourly price series, size solar and storage behind the meter against it, and tell you the payback - and I know how to check whether the site is big enough that its own load moves the price, in which case that number is wrong and I would model it differently."*

That is a more employable sentence than anything about optimal 2035 generation mixes, and you can now do all of it.

### Sources and notes
- The merit order here is a teaching stand-in, but its *shape* - a long cheap base and a thin, very expensive tail - is the important realistic feature. ERCOT's system-wide offer cap has been set as high as $5,000/MWh.
- ERCOT 4CP transmission cost allocation: see ERCOT and your utility's published 4CP guidance.
- The site, the investors and the 600 MW agreement are invented; the pattern of converting industrial sites with existing interconnection into data centres is not.
